# ntv_ddml

> 对应代码：`EconometricsCode/code_in_notes/Chap.10/ntv_ddml.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们加载俄罗斯 NTV 的地区-年份面板数据，并用 tsset 声明面板维度（地区 tik_id 与时间 year），为随后生成滞后变量做准备。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
tsset tik_id year

控制变量取护士数、医生数、人口与工资的一期滞后（L. 算子），滞后取值可以缓解同期反向因果的担忧。

In [ ]:
%%stata
// 控制变量
gen lag_nurses=L.nurses
gen lag_doctors_pc=L.doctors_pc
gen lag_pop=L.logpop
gen lag_wage=L.log_wage

我们把四个滞后控制与地区虚拟变量 i.region 一并收入局部宏 controls，供后续两个条件期望方程共同使用。

In [ ]:
%%stata
local controls "lag_doctors_pc lag_nurses lag_pop lag_wage i.region"

ddml init partial 初始化一个部分线性模型（partially linear model）：结果方程 Y=g(X)+θD+U，处理方程 D=m(X)+V，我们的目标参数是 θ。kfolds(10) 指定把样本随机分为 10 折做交叉拟合（cross-fitting），即每一折的残差化都只用其余 9 折训练出的模型完成，避免用自己的数据拟合自己所带来的过拟合偏差。

In [ ]:
%%stata
// DDML
ddml init partial, kfolds(10)

这一步为 E[Y|X] 指定机器学习算法：用 pystacked 调用随机森林（method(rf)，n_estimators(200) 即 200 棵树），以控制变量 X 预测结果变量 Votes_SPS_（支持 SPS 党的得票）。ddml 会据此在每一折外样本上生成 Y 的条件期望拟合值。

In [ ]:
%%stata
ddml E[Y|X]: pystacked Votes_SPS_  `controls', type(regress) method(rf) cmdopt1(n_estimators(200))

类似地，我们为 E[D|X] 指定随机森林回归：用同样的控制变量 X 预测处理变量 Watch_probit（地区收看 NTV 的强度）。对 Y 和 D 分别关于 X 残差化，正是 Frisch–Waugh–Lovell 思想在高维控制下的推广，也是双重机器学习"正交化"的关键一步。

In [ ]:
%%stata
ddml E[D|X]: pystacked Watch_probit `controls', type(regress) method(rf) cmdopt1(n_estimators(200))

ddml crossfit 执行交叉拟合：在每一折的留一样本上计算残差 Y-E[Y|X] 与 D-E[D|X]，由于 Neyman 正交性，这些残差对机器学习第一步的估计误差不敏感，从而允许我们用非参数方法灵活拟合烦杂函数 g(·)、m(·)。

In [ ]:
%%stata
ddml crossfit

最后用残差对残差做回归，得到处理效应 θ 的估计，并按地区聚类报告稳健标准误（cluster(region)）。这正等价于注释中所示的 reg Y1_pystacked_1 D1_pystacked_1——部分线性模型 DML 的第二阶段就是一个简单的一元回归。

In [ ]:
%%stata
ddml estimate,  cluster(region)